## Proyecto Práctico – Procesamiento de un Dataset con Spark utilizando RRDs
###### Contexto: El REMS o “Rover Environmental Monitoring Station” es una estación meteorológica en Marte creada para el rover Curiosity y provista por España y Finlandia. El REMS mide la humedad, la presión, la temperatura, la velocidad del viento y la radiación ultravioleta en Marte. Este proyecto español está liderado por el Centro Español de Astrobiología e incluye como socio al Instituto Meteorológico finlandés, aportando sensores de presión y humedad.

###### Dataset: El dataset contiene 10 columnas con los siguientes encabezados en orden de aparicion: earth_date_time,	sol_number,	max_ground_temp(°C),	min_ground_temp(°C),	max_air_temp(°C),	min_air_temp(°C),	mean_pressure(Pa),	sunrise	sunset,	UV_Radiation.
###### Objetivo: A partir del dataset con las observaciones del clima del planeta Marte recabados por el Mars Curiosity Rover, obtener los valores para las siguientes métricas:
1.	Temperatura máxima del suelo del planeta y día de la medición.
2.	Temperatura mínima del aire del planeta y día de la medición.
3.	Presión atmosférica promedio máxima obtenida y el valor del delta de temperatura del aire pare ese mismo día de la medición.
4.  Filtrar las observaciones con Radiación UV con la categoria "very_high" e indicar los valores de temperatura máxima registrada.

# 1. Configuración de PySpark en Google Colab

In [18]:
# Importamos la librería de apache spark
import pyspark as ps 
from pyspark.sql import SparkSession # Importamos SparkSesion 

print(f'Versión de Spark: {ps.__version__}')

Versión de Spark: 3.5.4


# 2. Creamos la sesión de Spark

In [19]:
# Primero, creamos una instancia de SparkContext
# Usamos SparkSession.builder para obtener un SparkContext
spark = SparkSession.builder \
    .appName("Rover_RDD") \
    .getOrCreate()

# 3. Crear el RDD inicial a partir del dataset y mostrar las primeras observaciones

In [20]:
# Definir la ruta de los archivos en Google Drive
input_path = 'C:\\Users\\demad\\Documents\\Master\\Spark\\Ficheros Input Notebooks\\Clase 1\\REMS_Mars_Dataset.csv'

# Leer el archivo CSV con Spark
lines = spark.sparkContext.textFile(input_path)

df = spark.read.format("csv") \
    .option("inferSchema", "true") \
    .option("header", "false") \
    .load(input_path)

# Mostrar el DataFrame
df.show()

+--------------------+--------+---+---+---+---+---+-------------------+-------------------+--------+
|                 _c0|     _c1|_c2|_c3|_c4|_c5|_c6|                _c7|                _c8|     _c9|
+--------------------+--------+---+---+---+---+---+-------------------+-------------------+--------+
|Earth, 2022-01-26...|Sol 3368| -3|-71| 10|-84|707|2025-06-09 05:25:00|2025-06-09 17:20:00|moderate|
|Earth, 2022-01-25...|Sol 3367| -3|-72| 10|-87|707|2025-06-09 05:25:00|2025-06-09 17:20:00|moderate|
|Earth, 2022-01-24...|Sol 3366| -4|-70|  8|-81|708|2025-06-09 05:25:00|2025-06-09 17:21:00|moderate|
|Earth, 2022-01-23...|Sol 3365| -6|-70|  9|-91|707|2025-06-09 05:26:00|2025-06-09 17:21:00|moderate|
|Earth, 2022-01-22...|Sol 3364| -7|-71|  8|-92|708|2025-06-09 05:26:00|2025-06-09 17:21:00|moderate|
|Earth, 2022-01-21...|Sol 3363| -8|-71|  8|-80|707|2025-06-09 05:26:00|2025-06-09 17:21:00|moderate|
|Earth, 2022-01-20...|Sol 3362| -4|-72|  5|-84|706|2025-06-09 05:26:00|2025-06-09 17:21:00|

# 4. Hacemos una funcion parser para extraer los datos

In [23]:
def parser(line):
    fields = line.split(',')
    earth_date_time = fields[1]
    sol_number = fields[2]
    max_ground_temp = float(fields[3])
    min_ground_temp = float(fields[4])
    max_air_temp = float(fields[5])
    min_air_temp = float(fields[6])
    mean_pressure = float(fields[7])
    sunrise = fields[8]
    sunset = fields[9]
    uv_radiation = fields[10]

    return (earth_date_time, max_ground_temp, min_ground_temp, max_air_temp, min_air_temp, mean_pressure, sunrise, sunset, uv_radiation )

# 5. Usamos el persist

In [24]:
parsed = lines.map(parser).persist() # Persiste el RDD en el storagee level MEMORY_AND_DISK por defecto.

# 6. Calculamos la temperatura máxima del suelo del planeta y día de la medición.




In [13]:
tempmax_rdd = parsed.map(lambda x: (x[1], x[0]))
sortedByAirMaxTemp = tempmax_rdd.sortByKey(ascending=False).collect()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 5.0 failed 1 times, most recent failure: Lost task 0.0 in stage 5.0 (TID 7) (LucentiaMSI2 executor driver): java.net.SocketException: Connection reset
	at java.base/sun.nio.ch.NioSocketImpl.implRead(NioSocketImpl.java:328)
	at java.base/sun.nio.ch.NioSocketImpl.read(NioSocketImpl.java:355)
	at java.base/sun.nio.ch.NioSocketImpl$1.read(NioSocketImpl.java:808)
	at java.base/java.net.Socket$SocketInputStream.read(Socket.java:966)
	at java.base/java.io.BufferedInputStream.fill(BufferedInputStream.java:244)
	at java.base/java.io.BufferedInputStream.read(BufferedInputStream.java:263)
	at java.base/java.io.DataInputStream.readInt(DataInputStream.java:393)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:774)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(Pytho

In [14]:
result = sortedByAirMaxTemp[0]
print(f"Earth Time: {result[1]}, Max. Soil Temperature: {result[0]} Deg. Cent.")

NameError: name 'sortedByAirMaxTemp' is not defined

#7. Calculamos la temperatura minima del aire del planeta y día de la medición.

In [15]:
temp_rdd = parsed.map(lambda x: (x[4], x[0]))
sortedByAirTemp = temp_rdd.sortByKey(ascending=True).collect()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 6.0 failed 1 times, most recent failure: Lost task 0.0 in stage 6.0 (TID 9) (LucentiaMSI2 executor driver): java.net.SocketException: Connection reset
	at java.base/sun.nio.ch.NioSocketImpl.implRead(NioSocketImpl.java:328)
	at java.base/sun.nio.ch.NioSocketImpl.read(NioSocketImpl.java:355)
	at java.base/sun.nio.ch.NioSocketImpl$1.read(NioSocketImpl.java:808)
	at java.base/java.net.Socket$SocketInputStream.read(Socket.java:966)
	at java.base/java.io.BufferedInputStream.fill(BufferedInputStream.java:244)
	at java.base/java.io.BufferedInputStream.read(BufferedInputStream.java:263)
	at java.base/java.io.DataInputStream.readInt(DataInputStream.java:393)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:774)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(Pytho

In [ ]:
result = sortedByAirTemp[0]
print(f"Earth Time: {result[1]}, Min. Air Temperature: {result[0]} Deg. Cent.")

# 8. Calculamos la presión atmosférica promedio maxima obtenida y el valor del delta de temperatura del aire pare ese mismo día de la medición.

In [16]:
pressure_rdd = parsed.map(lambda x: (x[0], (x[5], x[3] - x[4])))
sortedBypressure = pressure_rdd.map(lambda x: (x[1][0], (x[0],x[1][1]))).sortByKey(ascending=False).collect()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 7.0 failed 1 times, most recent failure: Lost task 0.0 in stage 7.0 (TID 11) (LucentiaMSI2 executor driver): java.net.SocketException: Connection reset
	at java.base/sun.nio.ch.NioSocketImpl.implRead(NioSocketImpl.java:328)
	at java.base/sun.nio.ch.NioSocketImpl.read(NioSocketImpl.java:355)
	at java.base/sun.nio.ch.NioSocketImpl$1.read(NioSocketImpl.java:808)
	at java.base/java.net.Socket$SocketInputStream.read(Socket.java:966)
	at java.base/java.io.BufferedInputStream.fill(BufferedInputStream.java:244)
	at java.base/java.io.BufferedInputStream.read(BufferedInputStream.java:263)
	at java.base/java.io.DataInputStream.readInt(DataInputStream.java:393)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:774)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(Pyth

In [ ]:
result = sortedBypressure[1]
print(f"Earth Time: {result[1][0]}, Max. Avg. Pressure: {result[0]} , Temperature: {result[1][1]} Deg. Cent.")

# 9. Filtramos las observaciones con Radiación UV con la categoria "very_high" e indicamos los valores de temperatura máxima registrada.

In [17]:
maxUvReadings = parsed.filter(lambda x: "very_high" in x[8])
dateMaxTemp = maxUvReadings.map(lambda x: (x[0],x[1]))
maxTemp = dateMaxTemp.reduceByKey(lambda x, y: max(x,y))
flipped_maxTemp = maxTemp.map(lambda x: (x[1], x[0]))
results = flipped_maxTemp.sortByKey(ascending=False).collect()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 8.0 failed 1 times, most recent failure: Lost task 1.0 in stage 8.0 (TID 14) (LucentiaMSI2 executor driver): java.net.SocketException: Connection reset
	at java.base/sun.nio.ch.NioSocketImpl.implRead(NioSocketImpl.java:328)
	at java.base/sun.nio.ch.NioSocketImpl.read(NioSocketImpl.java:355)
	at java.base/sun.nio.ch.NioSocketImpl$1.read(NioSocketImpl.java:808)
	at java.base/java.net.Socket$SocketInputStream.read(Socket.java:966)
	at java.base/java.io.BufferedInputStream.fill(BufferedInputStream.java:244)
	at java.base/java.io.BufferedInputStream.read(BufferedInputStream.java:263)
	at java.base/java.io.DataInputStream.readInt(DataInputStream.java:393)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:774)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(Pyth

In [ ]:
for result in results:
    print(f"Earth Time: {result[1]}, Max Temp.: {result[0]} Deg. Centigrades")